# DriveInsight

## About

DriveInsight is a large-scale, CCTV-based traffic dataset that bridges the gap between real-world traffic observations and simulation-ready scenarios. Scenarios are extracted from 24/7 CCTV camera feeds at urban intersections across more than 20 locations in multiple countries, then converted into standardized OpenSCENARIO (.xosc) format. Each scenario ships with a corresponding OpenDRIVE (.xodr) road network, weather metadata, and temporal annotations, making it directly compatible with simulation tools such as esmini, CARLA, and VTD.

The dataset covers a diverse range of traffic participants and is designed for trajectory forecasting, scenario-based ADAS validation, multi-agent interaction modelling, and long-tail behaviour analysis.

### How to Obtain the Dataset

The DriveInsight dataset is publicly available on GitHub. Clone the repository directly to get started:

```bash
git clone https://github.com/Drive1nsight/driveinsightD.git
cd driveinsightD
```

For extended intersections, additional scenario collections, and high-resolution CCTV source footage, submit a request via the project website at [https://driveinsight.io](https://driveinsight.io).

### How to Obtain the Map

Road network files in OpenDRIVE (.xodr) format are bundled with the dataset under each location folder:

```
driveinsightD/
└── database/
    └── <location_id>/
        ├── {id}_scenario.xosc          # OpenSCENARIO trajectory file
        ├── {id}_scenario_metadata.json # Scenario metadata
        ├── <location_id>.xodr          # OpenDRIVE road network
        └── <location_id>.osgb          # 3D environment model
```

No separate download is required — the map file for each location is located in the same folder as the scenario files.

### Citation

```latex
@article{zhdanov2025driveinsight,
  title={DriveInsight: CCTV-based dataset capturing real-world scenarios},
  author={Zhdanov, Pavlo and Yerumbayev, Sultan and Khan, Adil and Ram{\'i}rez Rivera, Ad{\'i}n},
  journal={IEEE Transactions on Circuits and Systems for Video Technology},
  year={2025},
  note={Under review}
}
```

In [1]:
%matplotlib inline
import warnings

warnings.filterwarnings("ignore")

import logging

logging.basicConfig(level=logging.WARNING)


import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import Point
import seaborn as sns

In [2]:
# Setting up parameters for matplotlib
mpl.rcParams.update(
    {
        "figure.dpi": 200,  # 200 for high quality
        "font.family": "DejaVu Sans Mono",
        "font.size": 6,
        "font.stretch": "semi-expanded",
        "animation.html": "jshtml",
        "animation.embed_limit": 5000,
        "axes.edgecolor": "black",
        "axes.linewidth": 0.8,
        "axes.grid": True,
        "grid.color": "#cccccc",
        "axes.facecolor": "white",
    }
)
sns.set_palette("Set2")

## Data Analysis

> This part is independently conducted by Tactics2D.

### Trajectory Types in DriveInsight

The DriveInsight dataset only includes `car` category for vehicles.

## Tactics2D Integration

### Dataset Preparation

You can place the DriveInsightD dataset in any directory of your choice. However, it’s important to maintain the original folder structure as provided in the dataset.

### Class Mapping

The `car` category in DriveInsightD is mapped to the `Vehicle` class in Tactics2D. This means that all trajectories labeled as `car` in the dataset will be treated as `Vehicle` objects within the Tactics2D framework.

### Parse and Replay Logs

To parse and replay the DriveInsightD scenarios, you can use the following code snippet:

In [3]:
%matplotlib notebook

from IPython.display import HTML
from matplotlib.animation import FuncAnimation

from tactics2d.dataset_parser import DriveInsightDParser
from tactics2d.renderer import MatplotlibRenderer
from tactics2d.sensor import BEVCamera

In [4]:
def render_scenario_animation(scenario: dict, resolution=(1200, 800), step_ms: int = 250):
    map_ = scenario["map"]
    participants = scenario["participants"]
    t_start, t_end = scenario["time_range"]

    # Re-key participants with integer IDs (BEVCamera requires int IDs)
    int_participants = {}
    for idx, (name, p) in enumerate(participants.items()):
        class_ = type(p)
        new_p = class_(
            id_=idx,
            type_=p.type_,
            length=p.length,
            width=p.width,
            trajectory=p.trajectory,
        )
        new_p.trajectory.id_ = idx
        int_participants[idx] = new_p

    all_ts = set()
    for p in int_participants.values():
        frames = getattr(p.trajectory, "frames", None)
        if frames is not None:
            all_ts.update(frames)
    times = sorted(t for t in all_ts if t_start <= t <= t_end)[:: max(1, step_ms // 50)]

    boundary = map_.boundary
    x_min, x_max, y_min, y_max = boundary
    camera_position = np.array([(x_min + x_max) / 2, (y_min + y_max) / 2])

    camera = BEVCamera(id_=0, map_=map_)
    prev_road_id_set = set()
    prev_participant_id_set = set()

    renderer = MatplotlibRenderer(
        xlim=(x_min, x_max), ylim=(y_min, y_max), resolution=resolution, auto_scale=True
    )
    fig = renderer.fig

    def update(t):
        nonlocal prev_road_id_set, prev_participant_id_set
        participant_ids = [pid for pid, p in int_participants.items() if p.is_active(t)]
        geometry_data, prev_road_id_set, prev_participant_id_set = camera.update(
            t,
            int_participants,
            participant_ids,
            prev_road_id_set,
            prev_participant_id_set,
            Point(camera_position),
        )
        renderer.update(geometry_data)
        renderer.ax.set_title(f"t = {t} ms  |  active: {len(participant_ids)}", fontsize=6)

    ani = FuncAnimation(fig, update, frames=times, interval=100, repeat=True)
    plt.close(fig)
    return HTML(ani.to_html5_video())

In [5]:
# Zlin, Czech Republic — scenario 11 (cz_zlin.xodr)
dataset_parser = DriveInsightDParser()
cz_folder = "../../tactics2d/data/trajectory_sample/driveinsightD/database/cz_zlin"
cz_scenario = dataset_parser.parse(
    scenario_id="11",
    folder=cz_folder,
    map_name="cz_zlin.xodr",
)
print("Zlin, Czech Republic — scenario 11")
print(f"  Participants : {len(cz_scenario['participants'])}")
print(f"  Lanes        : {len(cz_scenario['map'].lanes)}")
render_scenario_animation(cz_scenario)

Zlin, Czech Republic — scenario 11
  Participants : 21
  Lanes        : 69


In [6]:
# Taito, Japan — scenario 6 (jp_taito.xodr)
jp_folder = "../../tactics2d/data/trajectory_sample/driveinsightD/database/jp_taito"
jp_scenario = dataset_parser.parse(
    scenario_id="6",
    folder=jp_folder,
    map_name="jp_taito.xodr",
)
print("Taito, Japan — scenario 6")
print(f"  Participants : {len(jp_scenario['participants'])}")
print(f"  Lanes        : {len(jp_scenario['map'].lanes)}")
render_scenario_animation(jp_scenario)

Taito, Japan — scenario 6
  Participants : 13
  Lanes        : 79


In [7]:
# Coldwater, United States — scenario 4464 (usa_coldwater.xodr)
us_folder = "../../tactics2d/data/trajectory_sample/driveinsightD/database/us_coldwater"
us_scenario = dataset_parser.parse(
    scenario_id="4464",
    folder=us_folder,
    map_name="usa_coldwater.xodr",
)
print("Coldwater, United States — scenario 4464")
print(f"  Participants : {len(us_scenario['participants'])}")
print(f"  Lanes        : {len(us_scenario['map'].lanes)}")
render_scenario_animation(us_scenario)

Coldwater, United States — scenario 4464
  Participants : 16
  Lanes        : 234


## Appendix: Data Format

<div class="admonition note">
    <p class="admonition-title">Note</p>
    <p>
        This is a summary of the DriveInsight dataset format based on
        <a href="https://github.com/Drive1nsight/driveinsightD" target="_blank" rel="noopener noreferrer">
            the official repository
        </a>, provided here for reference purposes only.
    </p>
</div>

### Statistics

#### Coverage

Locations: 20+ urban intersections across multiple countries and continents  
Source footage: Hundreds of hours of 24/7 CCTV streams  
Scenario format: OpenSCENARIO (.xosc)  
Map format: OpenDRIVE (.xodr)

#### Agent Types and Count

| Agent Type   | Approximate Count |
|--------------|------------------|
| Car          | ~120,000         |
| Truck        | ~17,500          |
| Bus          | ~2,700           |
| Pedestrian   | ~55,000          |
| Bicycle      | ~34,000          |
| Motorcycle   | ~650             |

### Description of the Data and File Structure

Each scenario is stored as a pair of files under `database/<location_id>/`:

- **`{id}_scenario.xosc`**: OpenSCENARIO XML file defining all agent entities, their dimensions, and trajectory waypoints in world coordinates.
- **`{id}_scenario_metadata.json`**: JSON file containing scenario-level annotations:

```json
{
  "camera_name": "cz_zlin",
  "datetime_utc": "2025-10-16T18:21:45+00:00",
  "location_name": "Zlin, Czech Republic",
  "road_description": {
    "road_name": "Tr. Tomase Bati / Osvoboditelu",
    "road_type": "Urban Road",
    "road_condition": "Good",
    "traffic_density": "Moderate",
    "configuration": "crossing"
  },
  "weather": {
    "lat": 49.2256,
    "lon": 17.6687,
    "timezone": "Europe/Prague"
  }
}
```

- **`<location_id>.xodr`**: OpenDRIVE road network defining lane geometry, intersections, and road topology for the recording location.
- **`<location_id>.osgb`**: OpenSceneGraph binary 3D environment model for visualisation in esmini and compatible simulators.

### Simulator Compatibility

DriveInsight scenarios are compatible with the following industry-standard simulation tools:

- [esmini](https://github.com/esmini/esmini)
- [CARLA](https://carla.org)
- [VTD](https://www.mscsoftware.com/product/virtual-test-drive)